# Scanpy Extra: (For quick testing)

In [37]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/SCWF00021/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from scanpy_init_env import *
from scanpy_utils import *
from scanpy_gene_lists import *

# Set variables
resolutions = [0.1, 0.2, 0.3] 

# Check Vln clusters

In [ ]:
# Load L1 clusters
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)

logger.info("Processing L2 subclusters ...")
subcluster_dir = scanpy_dir + 'subclustering/'
cell_types = ['Glu-UL', 'Glu-DL', 'NPC', 'GABA']

umap_figs = []         
adata_dict = {}     

for cell_type in cell_types:
    logger.info(f"Loading L2 data for {cell_type} ...")
    adata_sub = sc.read(subcluster_dir + f'adata_{cell_type}_subclusters.h5ad')
    
    if 'subcluster' not in adata_sub.obs.columns:
        raise KeyError(f"'subcluster' column missing in L2 object for {cell_type}")
    
    adata_dict[cell_type] = adata_sub   # optional

# Plot details
fig = plt.figure(figsize=(10, 9.5))           
gs = gridspec.GridSpec(nrows=2, ncols=2,
                       figure=fig,
                       wspace=0.35, hspace=0.42)

# Titles / panel labels
panel_labels = ["A", "B", "C", "D"]

for i, cell_type in enumerate(cell_types):
    row = i // 2
    col = i % 2
    
    ax = fig.add_subplot(gs[row, col])
    
    adata = adata_dict[cell_type]   # or reload if memory is not an issue
    
    sc.pl.umap(
        adata,
        color="subcluster",     
        ax=ax,
        palette=None,               
        legend_loc="on data",      
        legend_fontsize=9,
        legend_fontoutline=2,
        frameon=False,
        title=cell_type, 
        show=False
    )
    
    # Panel label (A/B/C/D) in top-left of each UMAP
    ax.text(-0.03, 1.03, panel_labels[i],
            transform=ax.transAxes,
            fontsize=16, fontweight='bold', va='top', ha='left')

# Global title
fig.suptitle("Level 2 subclusters – UMAP by subcluster", fontsize=14, y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])   # leave space for suptitle if used

# Save high-resolution PDF 
output_pdf = "L2_subclusters_2x2_UMAP.pdf"
plt.savefig(output_pdf, dpi=300, bbox_inches="tight")
logger.info(f"Saved 2×2 UMAP figure to: {output_pdf}")


plt.show()